# Scenario 2: Website Analytics Debug
**Scenario:** The marketing team reports that the user engagement metrics code is showing impossible results (bounce rates over 100%, negative session times).

## Prompt
**Initial Prompt Strategy:**
> "I have a Python function for website analytics that is returning impossible results, like bounce rates over 100% and negative session times. 
> 
> Here is the problematic code:
> ```python
> def analyze_user_engagement(logs_df):
>     metrics = {
>         'bounce_rate': logs_df.groupby('session_id')['page_views'].apply(lambda x: x == 1).mean(),
>         'avg_session_time': logs_df.groupby('session_id')['duration'].sum(),
>         'pages_per_session': logs_df.groupby('session_id')['page_views'].mean()
>     }
>     device_metrics = logs_df.groupby('device_type').agg({'session_id': 'count', 'duration': 'mean', 'page_views': 'sum'})
>     return metrics, device_metrics
> ```
> Please debug the calculations to fix these errors. Also, add data validation to ensure no negative times are processed, implement proper time calculations, and create some summary visualizations."

## Generated Code

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_user_engagement(logs_df):
    # Data Validation
    if logs_df.empty:
        raise ValueError("Log dataframe is empty")
        
    # Filter out negative session times
    valid_logs = logs_df[logs_df['duration'] >= 0].copy()
    
    # Debugged calculations
    # 1. Bounce rate: % of sessions with exactly 1 total page view
    session_page_views = valid_logs.groupby('session_id')['page_views'].sum()
    bounce_rate = (session_page_views == 1).mean()
    
    # 2. Avg session time: Overall average of session durations
    session_durations = valid_logs.groupby('session_id')['duration'].sum()
    avg_session_time = session_durations.mean()
    
    # 3. Pages per session: Overall average
    pages_per_session = session_page_views.mean()
    
    metrics = {
        'bounce_rate': bounce_rate,
        'avg_session_time': avg_session_time,
        'pages_per_session': pages_per_session
    }
    
    # Fix device metrics: count unique sessions instead of total log rows
    device_metrics = valid_logs.groupby('device_type').agg({
        'session_id': 'nunique',
        'duration': 'mean',
        'page_views': 'sum'
    }).rename(columns={'session_id': 'unique_sessions'})
    
    # Visualization
    plt.figure(figsize=(10, 5))
    sns.barplot(x=device_metrics.index, y='unique_sessions', data=device_metrics)
    plt.title('Unique Sessions by Device Type')
    plt.ylabel('Sessions')
    plt.show()
    
    return metrics, device_metrics

## Follow-up Prompts
**Refinement & Testing:**
The initial response fixed the obvious bugs (returning Series instead of floats, handling negative durations by dropping them). However, silently dropping bad data isn't always best, and time calculations might be better served by converting to `pd.Timedelta`. 

> *"The debugged code is much better, but I'd like to refine the validation and time handling:*
> *1. Instead of just filtering out negative durations silently, log a warning with the count of invalid records before removing them.*
> *2. The 'duration' is in seconds. Please convert the `avg_session_time` metric into a formatted `MM:SS` string rather than just a float.*
> *3. Add a visualization showing the distribution of session durations (in minutes) to spot outliers."*

## Final Solution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging

# Configure logging
logging.basicConfig(level=logging.WARNING)

def analyze_user_engagement_optimized(logs_df):
    df = logs_df.copy()
    
    # 1. Advanced Data Validation with Logging
    if 'duration' not in df.columns or 'page_views' not in df.columns:
        raise ValueError("Missing required columns in dataset.")
        
    negative_durations = (df['duration'] < 0).sum()
    if negative_durations > 0:
        logging.warning(f"Found {negative_durations} records with negative duration. These will be removed.")
        df = df[df['duration'] >= 0]
        
    invalid_page_views = (df['page_views'] <= 0).sum()
    if invalid_page_views > 0:
        logging.warning(f"Found {invalid_page_views} records with 0 or negative page views. Removing.")
        df = df[df['page_views'] > 0]

    # Group by session once to optimize calculations
    session_grp = df.groupby('session_id')
    session_data = session_grp.agg({
        'page_views': 'sum',
        'duration': 'sum'
    })

    # 2. Corrected Calculations
    # Bounce Rate: Sessions with exactly 1 total page view
    bounce_rate = (session_data['page_views'] == 1).mean()
    
    # Pages per Session
    pages_per_session = session_data['page_views'].mean()
    
    # 3. Proper Time Handling (Convert Seconds to MM:SS format)
    avg_seconds = session_data['duration'].mean()
    minutes = int(avg_seconds // 60)
    seconds = int(avg_seconds % 60)
    avg_session_time_formatted = f"{minutes:02d}:{seconds:02d}"

    metrics = {
        'bounce_rate': round(bounce_rate, 4),
        'avg_session_time_seconds': round(avg_seconds, 2),
        'avg_session_time_formatted': avg_session_time_formatted,
        'pages_per_session': round(pages_per_session, 2)
    }
    
    # Device Metrics (Ensure we count unique sessions!)
    device_metrics = df.groupby('device_type').agg({
        'session_id': 'nunique',  # Correct fix: count unique sessions, not rows
        'duration': 'mean',       # Mean duration per interaction
        'page_views': 'sum'
    })
    
    # 4. Visualizations
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Sessions by Device
    sns.barplot(x=device_metrics.index, y='session_id', data=device_metrics, ax=axes[0], palette='viridis')
    axes[0].set_title('Unique Sessions by Device Type')
    axes[0].set_ylabel('Unique Sessions')
    
    # Plot 2: Session Duration Distribution (converted to minutes)
    session_data['duration_minutes'] = session_data['duration'] / 60
    sns.histplot(session_data['duration_minutes'], bins=30, kde=True, ax=axes[1], color='coral')
    axes[1].set_title('Distribution of Session Durations')
    axes[1].set_xlabel('Duration (Minutes)')
    axes[1].set_xlim(0, session_data['duration_minutes'].quantile(0.95)) # Zoom in on 95th percentile to hide extreme outliers
    
    plt.tight_layout()
    plt.show()

    return metrics, device_metrics

# Example usage:
# metrics_summary, device_summary = analyze_user_engagement_optimized(logs_df)

### Discussion Questions to Answer:

**1. How did different prompts approach error identification?**
* A general prompt ("Fix this code") led the AI to correctly identify that `.sum()` and `.mean()` were returning Pandas Series rather than single scalar values. It also realized that calculating bounce rate using `apply(lambda x: x==1)` without aggregating first was incorrect.
* When pushed with a more specific prompt that mentioned *why* the data was impossible (negative session times), the AI actively began looking for data anomalies and wrote logic to address them rather than just fixing syntax errors.

**2. What validation methods were suggested?**
* The initial output used a simple, silent boolean mask: `logs_df[logs_df['duration'] >= 0]`. While this stops the calculation errors, it hides data ingestion issues from the team.
* Through iterative prompting, the validation became more robust: checking for required columns, and using Python's `logging` module to output warnings describing exactly how many rows of data were invalid (e.g., negative duration or zero page views) before dropping them.

**3. How was time handling implemented?**
* At first, time was just treated as a raw integer/float (seconds) and mathematically averaged.
* After refinement, the AI calculated the total seconds, then used floor division `//` and modulo `%` to format the average session time into a much more readable `MM:SS` string (e.g., `03:45`). Additionally, for the visualization, it intelligently converted the duration into minutes by dividing by 60 and applied a 95th percentile `xlim` to ensure the histogram wasn't skewed by extreme outliers.